In [1]:
import os
import pandas as pd
import re
from pathlib import Path
import ast
import re
import numpy as np
from sklearn.metrics import cohen_kappa_score
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import json
import time
import pickle

path = "C:\\Users\\USER\\Documents\\empirical_evidence_corren"

# Set the project root directory
PROJECT_ROOT = Path(path)

# Function to construct paths relative to the project root
def project_path(*args):
    return PROJECT_ROOT.joinpath(*args)

text_file_path = project_path("scripts", "claimbuster_API_key.txt")

# Open and read the text file, saving the content as a string
with open(text_file_path, 'r') as file:
    file_text = file.read()  # Read the entire content into a string


In [33]:
df_predictions = pd.read_csv(project_path('data','merged_hearings_roberta_predict.csv'))
df_predictions_withclaimbuster = pd.read_csv(project_path('data','df_predictions_withclaimbuster.csv'))

In [38]:
df_predictions = df_predictions.sample(1500)

In [39]:
import requests
import pandas as pd
import time
import urllib.parse

# Assuming df_predictions is your DataFrame and file_text contains your API key
api_key = file_text
headers = {"x-api-key": api_key}

def get_claimbuster_score(sentence):
    try:
        encoded_sentence = urllib.parse.quote(sentence, safe='')
        url = f"https://idir.uta.edu/claimbuster/api/v2/score/text/{encoded_sentence}"
        response = requests.get(url, headers=headers, timeout=10)

        if response.status_code == 200:
            data = response.json()
            return data['results'][0]['score']
            print("completed sentence")
        else:
            print(f"Error {response.status_code} for sentence: {sentence}")
            return f"error_{response.status_code}"
    except requests.exceptions.RequestException as e:
        print(f"Request exception for sentence: {sentence} | Error: {e}")
        return "request_exception"
    except Exception as e:
        print(f"Unexpected exception for sentence: {sentence} | Error: {e}")
        return "unknown_error"

# Optionally add a sleep if you're hitting rate limits
scores = []
for sentence in df_predictions['target_sentence']:
    scores.append(get_claimbuster_score(sentence))
    time.sleep(0.5)  # adjust if needed

df_predictions['claimbuster_score'] = scores

Error 404 for sentence: And were the banks more willing to provide checking accounts and/or wire transfers to money service businesses serving vulnerable nations because of your statement, as far as you are aware?
Error 404 for sentence: You heard a suggestion that Georgia Power/Southern Co. estimates a $15 to $20-million-a-year savings from the loan guarantee.
Error 404 for sentence: We thank you for your service both before but particularly after 9/11, where you brought--it was such a tragedy--brought this country together, sir.
Error 404 for sentence: That being said, in Texas, as you know, we do have the Gulf of Mexico where the Air/Marine is at, the Coast Guard is at, but we also have our local folks also, the sheriff also, Sheriff Garcia from Harris County.
Error 404 for sentence: The results of this effort must be ready soon if we are to make the 2016 launch window and to enable the NASA/ESA partnership to move forward.    
Error 404 for sentence: On page 4, note 4, specifically

In [40]:
# Convert string '[0.9948 0.0051]' to a numpy array, then extract the second value
def extract_empirical_prob(prob_str):
    try:
        arr = np.fromstring(prob_str.strip('[]'), sep=' ')
        return arr[1] if len(arr) > 1 else None
    except Exception as e:
        print(f"Error parsing: {prob_str} | {e}")
        return None

# Apply the function to create the new column
df_predictions['empirical_prob'] = df_predictions['probabilities'].apply(extract_empirical_prob)


In [41]:
df_predictions['claimbuster_score'] 

5813    0.473553
2491    0.220643
1028    0.112249
5692    0.613307
991     0.797507
          ...   
4391    0.267201
538      0.39136
894     0.117902
1118    0.250255
1916    0.656261
Name: claimbuster_score, Length: 1500, dtype: object

In [42]:
df_predictions = df_predictions[df_predictions['claimbuster_score'] != "error_404"]
df_predictions = df_predictions[df_predictions['claimbuster_score'] != "request_exception"]

# Step 2: Calculate the Pearson correlation between 'empirical_prob' and 'claimbuster_score'
correlation = df_predictions['empirical_prob'].corr(df_predictions['claimbuster_score'])

print(f"Pearson correlation: {correlation:.4f}")

Pearson correlation: 0.5810


In [45]:
df_predictions_withclaimbuster = df_predictions
df_predictions_withclaimbuster.to_csv(project_path("data", "df_predictions_withclaimbuster.csv"), index=False)